Calculate AMI Score

In [12]:
import pandas as pd
from sklearn.metrics import adjusted_rand_score
from sklearn.metrics import adjusted_mutual_info_score

input_file2 = "input_ami_ari.xlsx"
df2 = pd.read_excel(input_file2)

ari = adjusted_rand_score(
    df2["cluster_id"],
    df2["label_sme"]
)

ami = adjusted_mutual_info_score(
    df2["cluster_id"],
    df2["label_sme"]
)

print("ARI:", ari)
print("AMI:", ami)

ARI: 0.49371204519836337
AMI: 0.7907362110483438


Calculate F1 Score

In [15]:
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# ==========================================================
# 1. Load data
# ==========================================================
input_file = "input_ami_ari.xlsx"

df = pd.read_excel(input_file)


# ==========================================================
# 2. Clean label columns
# ==========================================================

# IMPORTANT:
# Drop actual missing values BEFORE converting to string.
df = df.dropna(
    subset=["label_prediction", "label_sme"]
).copy()

# Convert to string and remove leading/trailing spaces
df["label_prediction"] = (
    df["label_prediction"]
    .astype(str)
    .str.strip()
)

df["label_sme"] = (
    df["label_sme"]
    .astype(str)
    .str.strip()
)

# Remove textual representations of missing values
invalid_values = [
    "",
    "nan",
    "none",
    "null",
    "n/a",
    "na"
]

df = df[
    ~df["label_prediction"].str.lower().isin(invalid_values)
    &
    ~df["label_sme"].str.lower().isin(invalid_values)
].copy()


# ==========================================================
# 3. Get all unique intent labels
# ==========================================================

# IMPORTANT:
# Use the UNION of prediction and SME labels.
all_labels = sorted(
    set(df["label_prediction"].unique())
    |
    set(df["label_sme"].unique())
)


# ==========================================================
# 4. Overall Multiclass Evaluation
# ==========================================================
def evaluate_overall(data):

    y_true = data["label_sme"]
    y_pred = data["label_prediction"]

    # ------------------------------------------------------
    # Accuracy
    # ------------------------------------------------------
    accuracy = accuracy_score(
        y_true,
        y_pred
    )

    # ------------------------------------------------------
    # Macro Precision
    # ------------------------------------------------------
    precision_macro = precision_score(
        y_true,
        y_pred,
        labels=all_labels,
        average="macro",
        zero_division=0
    )

    # ------------------------------------------------------
    # Macro Recall
    # ------------------------------------------------------
    recall_macro = recall_score(
        y_true,
        y_pred,
        labels=all_labels,
        average="macro",
        zero_division=0
    )

    # ------------------------------------------------------
    # Macro F1
    # ------------------------------------------------------
    f1_macro = f1_score(
        y_true,
        y_pred,
        labels=all_labels,
        average="macro",
        zero_division=0
    )

    # ------------------------------------------------------
    # Weighted Precision
    # ------------------------------------------------------
    precision_weighted = precision_score(
        y_true,
        y_pred,
        labels=all_labels,
        average="weighted",
        zero_division=0
    )

    # ------------------------------------------------------
    # Weighted Recall
    # ------------------------------------------------------
    recall_weighted = recall_score(
        y_true,
        y_pred,
        labels=all_labels,
        average="weighted",
        zero_division=0
    )

    # ------------------------------------------------------
    # Weighted F1
    # ------------------------------------------------------
    f1_weighted = f1_score(
        y_true,
        y_pred,
        labels=all_labels,
        average="weighted",
        zero_division=0
    )

    # ------------------------------------------------------
    # Micro F1
    # ------------------------------------------------------
    f1_micro = f1_score(
        y_true,
        y_pred,
        labels=all_labels,
        average="micro",
        zero_division=0
    )

    # ------------------------------------------------------
    # Exact Agreement
    # ------------------------------------------------------
    exact_agreement = (
        y_true == y_pred
    ).mean()

    return {
        "n_records": len(data),

        "accuracy": accuracy,

        "precision_macro": precision_macro,
        "recall_macro": recall_macro,
        "f1_macro": f1_macro,

        "precision_weighted": precision_weighted,
        "recall_weighted": recall_weighted,
        "f1_weighted": f1_weighted,

        "f1_micro": f1_micro,

        "exact_agreement": exact_agreement
    }


# ==========================================================
# 5. Overall Evaluation
# ==========================================================
overall_result = evaluate_overall(df)

overall = pd.DataFrame(
    [overall_result]
)

overall.insert(
    0,
    "level",
    "Overall"
)


# ==========================================================
# 6. Per Cluster Evaluation
# ==========================================================

cluster_results = []

for cluster, group in df.groupby(
    "cluster_id",
    dropna=False
):

    result = evaluate_overall(group)

    result["cluster_id"] = cluster

    cluster_results.append(result)


cluster_df = pd.DataFrame(
    cluster_results
)

cluster_df = cluster_df[
    [
        "cluster_id",
        "n_records",
        "accuracy",

        "precision_macro",
        "recall_macro",
        "f1_macro",

        "precision_weighted",
        "recall_weighted",
        "f1_weighted",

        "f1_micro",
        "exact_agreement"
    ]
]


# ==========================================================
# 7. Per Prediction Label Evaluation
# ==========================================================
# IMPORTANT:
# Evaluate each prediction label against ALL records
# using One-vs-Rest.
#
# For a particular label:
#
# TP = prediction == label AND SME == label
# FP = prediction == label AND SME != label
# FN = prediction != label AND SME == label
# TN = prediction != label AND SME != label
# ==========================================================

prediction_results = []

for label in all_labels:

    # ------------------------------------------------------
    # Convert multiclass problem into binary OvR problem
    # ------------------------------------------------------
    y_true = (
        df["label_sme"] == label
    ).astype(int)

    y_pred = (
        df["label_prediction"] == label
    ).astype(int)

    # ------------------------------------------------------
    # Accuracy
    # NEW: ACCURACY PER PREDICTION LABEL
    # ------------------------------------------------------
    accuracy = accuracy_score(
        y_true,
        y_pred
    )

    # ------------------------------------------------------
    # Precision
    # ------------------------------------------------------
    precision = precision_score(
        y_true,
        y_pred,
        zero_division=0
    )

    # ------------------------------------------------------
    # Recall
    # ------------------------------------------------------
    recall = recall_score(
        y_true,
        y_pred,
        zero_division=0
    )

    # ------------------------------------------------------
    # F1
    # ------------------------------------------------------
    f1 = f1_score(
        y_true,
        y_pred,
        zero_division=0
    )

    # ------------------------------------------------------
    # Confusion Matrix Components
    # ------------------------------------------------------

    tp = (
        (y_true == 1)
        &
        (y_pred == 1)
    ).sum()

    fp = (
        (y_true == 0)
        &
        (y_pred == 1)
    ).sum()

    fn = (
        (y_true == 1)
        &
        (y_pred == 0)
    ).sum()

    tn = (
        (y_true == 0)
        &
        (y_pred == 0)
    ).sum()

    # ------------------------------------------------------
    # Number of predicted instances
    # ------------------------------------------------------
    n_predicted = (
        y_pred == 1
    ).sum()

    # ------------------------------------------------------
    # Number of SME/reference instances
    # ------------------------------------------------------
    n_sme = (
        y_true == 1
    ).sum()

    # ------------------------------------------------------
    # Exact correct predictions for this label
    # ------------------------------------------------------
    n_correct = tp

    prediction_results.append({

        "label_prediction": label,

        "n_predicted": n_predicted,

        "n_sme": n_sme,

        "n_correct": n_correct,

        # NEW
        "accuracy": accuracy,

        "precision": precision,

        "recall": recall,

        "f1_score": f1,

        "TP": tp,

        "TN": tn,

        "FP": fp,

        "FN": fn
    })


label_df = pd.DataFrame(
    prediction_results
)

# Sort by F1 descending
label_df = label_df.sort_values(
    by="f1_score",
    ascending=False
).reset_index(drop=True)


# ==========================================================
# 8. Per SME Label Evaluation
# ==========================================================
# IMPORTANT:
# Again, use One-vs-Rest across the ENTIRE dataset.
#
# SME label = reference / ground truth
# Prediction label = model/algorithm prediction
# ==========================================================

sme_results = []

for label in all_labels:

    # ------------------------------------------------------
    # Binary representation for this SME label
    # ------------------------------------------------------
    y_true = (
        df["label_sme"] == label
    ).astype(int)

    y_pred = (
        df["label_prediction"] == label
    ).astype(int)

    # ------------------------------------------------------
    # Accuracy
    # NEW: ACCURACY PER SME LABEL
    # ------------------------------------------------------
    accuracy = accuracy_score(
        y_true,
        y_pred
    )

    # ------------------------------------------------------
    # Precision
    # ------------------------------------------------------
    precision = precision_score(
        y_true,
        y_pred,
        zero_division=0
    )

    # ------------------------------------------------------
    # Recall
    # ------------------------------------------------------
    recall = recall_score(
        y_true,
        y_pred,
        zero_division=0
    )

    # ------------------------------------------------------
    # F1
    # ------------------------------------------------------
    f1 = f1_score(
        y_true,
        y_pred,
        zero_division=0
    )

    # ------------------------------------------------------
    # Confusion Matrix Components
    # ------------------------------------------------------

    tp = (
        (y_true == 1)
        &
        (y_pred == 1)
    ).sum()

    fp = (
        (y_true == 0)
        &
        (y_pred == 1)
    ).sum()

    fn = (
        (y_true == 1)
        &
        (y_pred == 0)
    ).sum()

    tn = (
        (y_true == 0)
        &
        (y_pred == 0)
    ).sum()

    # ------------------------------------------------------
    # SME support
    # ------------------------------------------------------
    n_sme = (
        y_true == 1
    ).sum()

    # ------------------------------------------------------
    # Number of predictions for this label
    # ------------------------------------------------------
    n_predicted = (
        y_pred == 1
    ).sum()

    # ------------------------------------------------------
    # Correctly classified instances
    # ------------------------------------------------------
    n_correct = tp

    sme_results.append({

        "label_sme": label,

        "n_sme": n_sme,

        "n_predicted": n_predicted,

        "n_correct": n_correct,

        # NEW
        "accuracy": accuracy,

        "precision": precision,

        "recall": recall,

        "f1_score": f1,

        "TP": tp,

        "TN": tn,

        "FP": fp,

        "FN": fn
    })


sme_df = pd.DataFrame(
    sme_results
)

# Sort by F1 descending
sme_df = sme_df.sort_values(
    by="f1_score",
    ascending=False
).reset_index(drop=True)


# ==========================================================
# 9. Round Numerical Results
# ==========================================================

# Overall
overall = overall.round(3)

# Cluster
cluster_df = cluster_df.round(3)

# Prediction label
label_df = label_df.round(3)

# SME label
sme_df = sme_df.round(3)


# ==========================================================
# 10. Save Results to Excel
# ==========================================================

output_file = "output_v7.xlsx"

with pd.ExcelWriter(
    output_file,
    engine="openpyxl"
) as writer:

    overall.to_excel(
        writer,
        sheet_name="Overall",
        index=False
    )

    cluster_df.to_excel(
        writer,
        sheet_name="Per_Cluster",
        index=False
    )

    label_df.to_excel(
        writer,
        sheet_name="Per_Prediction_Label",
        index=False
    )

    sme_df.to_excel(
        writer,
        sheet_name="Per_SME_Label",
        index=False
    )


print(
    f"Evaluation saved to {output_file}"
)

Evaluation saved to output_v7.xlsx
